# Testing

**Testování je důležitou součástí vývoje SW**.


- Testy ověřují, že program zůstává funkční i po změnách kódu
- Přinášejí jistotu při refactoringu
- Slouží jako ukázky použití, tedy zčásti jako dokumentace

## Dělení testů

1. **unit testing:** testování "nejmenších" testovatelných jednotek, izolovaných od zbytku,
1. **integration testing:** kontrola, že komponenty programu fungují pohromadě,
1. **end-to-end testing:** kontrola aplikace od začátku do konce,
1. **performance testing:** měření výkonu,
1. **functional testing:** testování konkrétních features

Unit testy jsou obvykle nejlevnější a nejrychlejší, ale samostatně nejsou nutně dostačující.

## Unit tests - motivace

Chceme něco sofistikovanějšího než:

- náhodné printy napříč zdrojovým kódem
- zakomentované kusy testovacího kódu
- závislost na magic constants

In [ ]:
import time

def compute_discounted_price(price):
    # print(f"input price: {price}")

    discount = 0.2137

    # evening bonus
    if time.localtime().tm_hour > 20:
        discount += 0.05

    result = price * (1 - discount)

    if price == 123:
        return 42

    # Commented-out "test"
    # assert result >= 0

    # print(f"{result}")
    return result

**Co je špatně?**

- zakomentované debugovací printy
- Magic constant neznámého významu
- skrytá závislost na `localtime`
- podivný ad hoc testovací hack (`if price == 123`)
- zakomentovaný test (`assert results >= 0`)


**Funkce prakticky netestovatelná**

- chybí kontrola nad časem -> nekonzistentní výsledky → tzv. *flaky tests*
- těžko ověřitelné chování: všudypřítomné printy
- je těžké porozumět očekávanému výstupu: magic constants
- je těžké porozumět správnému chování: skryté hacky

In [ ]:
def compute_discounted_price(price, base_discount, is_evening=False):
    if price < 0:
        raise ValueError("price must be non-negative")

    discount = base_discount
    if is_evening:
        discount += 0.05

    return price * (1 - discount)

### Jak psát testovatelný kód?

Je to dovednost, kterou člověk získá zkušeností.

Obecné principy:

- separation of responsibilities: jedna funkce má jeden účel, dělá jednu věc
- explicit dependencies: závislosti jsou explicitně uvedeny (např. jako parametry)
- dependency injection: předávání collaborators, namísto hardcoded collaborators
- pure functions: preferujeme funkce bez side effects, které jsou deterministické: se stejnými parametry vracejí stejné výsledky
- minimum skrytého globálního stavu
- minimum I/O uvnitř logiky: např. načítání dat oddělíme od nakládání s nimi
- navracení hodnot namísto tisku (print)


Kód psaný s ohledem na testování:

- bývá čitelnější
- modulární

Obrovské funkce (monolitické) se špatně testují.

## `unittest`

**Varování:** `unittest` a Jupyter-Book si moc dobře nerozumí - výstupy pod některými testy jsou podivné/rozbité. Než to opravím, doporučuji si to vyzkoušet ručně.

Základními stavebními kameny balíku `unittest` jsou čtyři koncepty:

- *test fixture* (někdy test context) reprezentuje přípravu všeho, co je na daný test třeba (načtení dat, připojení, příprava directory structure)
- *test case* testovací jednotka. Typicky určena pro testování jednoho kusu kódu (unit), obsahuje řadu konkrétních testů.
- *test suite* agreguje testy do větších celků
- *test runner* zajišťuje spouštění testů.

Začneme jednoduchou funkcí:

In [2]:
def add(x, y):
    return x + y

Balík `unittest` dělí testy do jednotlivých *test cases*, které definujeme jako potomka třídy `unittest.TestCase`. V jednom `TestCase` poté můžeme definovat libovolné množství metod. Metody, jejichž název má prefix `test_` pak reprezentují jednotlivé testy. Jeden `TestCase` tak může obsahovat více testů. Typicky se do `TestCase` snažím sdružit nějak příbuzné testy, tj. např. testy jednoho objektu, funkce apod.

Klíčovým prvkem každého testu je kontrola, zda testovaný kód dává správný výstup. K takovým kontrolám typicky slouží funkce typu `assert`, které zkontrolují, zda je jejich argument pravdivý. Není-li, vyvolají výjimku typu `AssertionError`. Hle:

In [1]:
try:
    assert(1 == 2)
except AssertionError:
    print("It failed, because 1 is not equal to 2")

It failed, because 1 is not equal to 2


In [4]:
assert(add(1, 2) == 2)

AssertionError: 

Balík `unittest` přináší ve třídě `unittest.TestCase` řadu metod, které implementují `assert` pro specifické případy (např. porovnávání floatů, kolekcí atd.). Použití těchto metod je výhodné - šetří práci, protože už za nás řeší logiku složitějších asserts.

**Poznámka:** V prostředí jupyter musí být u ukázek `unittest.main(argv=[''], exit=False)`. Za normálních okolností pro stadardní spouštění testů to není třeba.

In [7]:
import unittest

class TestAdd(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(1, 2), 3)

unittest.main(defaultTest="TestAdd", argv=[''], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


In [15]:
import unittest
import math
class TestAdd(unittest.TestCase):
    def test_add(self):
        # self.assertEqual(add(0.1, 0.2), 0.3)
        # self.assertTrue(math.fabs(add(0.1, 0.2)-0.3) < 1e-14)
        self.assertAlmostEqual(add(0.1, 0.2), 0.3)

    def test_subtract(self):
        self.assertAlmostEqual(add(2, 1), 2)

unittest.main(defaultTest="TestAdd", argv=[''], exit=False)

.F
FAIL: test_subtract (__main__.TestAdd)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/974964612.py", line 10, in test_subtract
    self.assertAlmostEqual(add(2, 1), 2)
AssertionError: 3 != 2 within 7 places (1 difference)

----------------------------------------------------------------------
Ran 2 tests in 0.010s

FAILED (failures=1)


### Parametrizované testy

In [18]:
import unittest

class TestAdd(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(1, 2), 3)
        self.assertEqual(add(-1, 2), 1)

unittest.main(defaultTest="TestAdd", argv=[''], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.005s

OK


- Lepší je metoda`.subTest`
- Přehlednější.
- Když jeden subtest selže, ostatní pokračují

In [19]:
import unittest
from itertools import product

class TestAdd(unittest.TestCase):
    def test_add(self):
        for a, b in product(range(5), range(6)):
            with self.subTest(a=a, b=b):
                if a == 1 and b >= 4:
                    self.assertEqual(add(a, b), a)
                else:
                    self.assertEqual(add(a, b), a+b)


unittest.main(defaultTest="TestAdd", argv=[''], exit=False)


FAIL: test_add (__main__.TestAdd) (a=1, b=4)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/3469638254.py", line 9, in test_add
    self.assertEqual(add(a, b), a)
AssertionError: 5 != 1

FAIL: test_add (__main__.TestAdd) (a=1, b=5)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/3469638254.py", line 9, in test_add
    self.assertEqual(add(a, b), a)
AssertionError: 6 != 1

----------------------------------------------------------------------
Ran 1 test in 0.002s

FAILED (failures=2)


### Více testů v TestCase

In [ ]:
import unittest

def add(x, y):
    return x + y

class TestAdd(unittest.TestCase):
    def test_add_int(self):
        self.assertEqual(add(1, 2), 3)

    def test_add_str(self):
        self.assertEqual(add("a", "b"), "ab")

    def test_add_float(self):
        self.assertAlmostEqual(add(0.1, 0.2), 0.3)

unittest.main(defaultTest="TestAdd", argv=[''], exit=False)

## Test discovery, organizace, pokrytí

Testy obyvkle umísťujeme do složky `tests` vedle balíčku.

```bash
mipy
|-- mipy
|    |-- math
|    |    |-- __init__.py
|    |    |-- other.py
|    |    |-- primes.py
|    |-- __init__.py
|    |-- __main__.py
|    |-- utils.py
|-- tests
|    |-- __init__.py
|    |-- test_math_other.py
|    |-- test_primes.py
|    |-- test_utils.py
|-- pyproject.toml
```
- členíme do souborů s prefixem `test_`
- obvykle jeden soubor na jeden modul v balíku
- každý soubor může obsahovat více `TestCase`


Nejsnazší způsob jak spustit testy je z top adresáře příkazem
```bash
python3 -m unittest
```
- první běží *test discovery*,
- projde rekurzivně aktuální složku a vyhledá všechny testy
- z nalezených testů sestaví `TestSuite`
- `TestRunner` spouští testy z `TestSuite`
- Testy lze do `TestSuite` přidávat i manuálně, ale o tom jindy.


**Tip:**

Užitečné spouštět testy s přepínačem `-v` (nebo `--verbose`) -> detailnější informace o průběhu testu.
```bash
$ python3 -m unittest -v tests/test_primes.py

test_called (tests.test_primes.TestPrimes) ... ok
test_primes (tests.test_primes.TestPrimes) ... ok
test_some (tests.test_primes.TestPrimes) ... ok
test_zero (tests.test_primes.TestPrimes) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK
```

Většina IDE umí testy detekovat, spouštět a zpracovávat jejich výstup, díky čemuž je používání testů velmi komfortní.


### Setup a Teardown

- některé testy vyžadují data
- je praktické přípravu dat oddělit od testu a případně zajistit, aby se data nenačítala zbytečně dvakrát

Můžeme použít speciální metody:

- metody `.setUp` a `.tearDown` se spouští před každým testem
- metody `.setUpClass` a `.tearDownClass` jednou pro daný `TestCase`.

Statistický příklad

In [ ]:
import unittest

def mean(x):
    return sum(x) / len(x)

def median(x):
    i = len(x) // 2 - 1

    if len(x) % 2 == 0:
        return (x[i] + x[i+1]) / 2
    else:
        return x[i]
    

class TestStat(unittest.TestCase):
    def setUp(self):
        print("setting up")
        self.data = list(range(1, 101))

    def test_mean(self):
        self.assertAlmostEqual(mean(self.data), 50.5)
    
    def test_empty(self):
        self.assertRaises(ZeroDivisionError, mean, [])

    def test_median(self):
        self.assertAlmostEqual(median(self.data), 50.5)

class TestStat2(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        print("setting up 2")
        cls.data = list(range(1, 101))

    def test_mean(self):
        self.assertAlmostEqual(mean(self.data), 50.5)
    
    def test_empty(self):
        self.assertRaises(ZeroDivisionError, mean, [])

    def test_median(self):
        self.assertAlmostEqual(median(self.data), 50.5)


unittest.main(argv=[''], exit=False)

**Otázka:** Je medián dobře otestovaný?

### Mocking and patching

- *mocking* je užitečná technika sloužící k nahrazení některých objektů falešnými mock objects)
- to umožňuje izolované testování nezávislé na nahrazovaných objektech
- *mocking* označuje vytváření falešného, zástupného objektu
- *patching* značí jeho dynamické podstrčení za objekt skutečný. 

#### Mocking

V příkladu níže:

- obtížné otestovat, za funkce `add` volá správně `.log`, aniž bychom se podívali do logu

In [20]:
class Logger:
    def log(self, message):
        print(f"Logging: {message}")

class Calculator:
    def __init__(self):
        self.logger = Logger()

    def add(self, a, b):
        result = a + b
        self.logger.log(f"Added {a} to {b} got {result}")
        return result

Nahradíme-li logger mock objektem, můžeme to přímo testovat

In [23]:
import unittest
from unittest.mock import Mock

class TestCalculator(unittest.TestCase):
    def test_addition(self):
        calculator = Calculator()
        calculator.logger = Mock()

        result = calculator.add(1, 2)

        self.assertEqual(result, 3)
        calculator.logger.log.assert_called_once_with("Added 1 to 2 got 3")

unittest.main(defaultTest="TestCalculator", argv=[''], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.004s

OK


- Testování `Calculator` je tak izolováno od `Logger`
- navíc nejsme závislí na side effects objektu `Logger` (např. na vytváření nějakých logfiles apod.)

Ještě lepší, když využijeme dependency injection+inversion, tj. objektu Calculator předáme hotový logger.

In [ ]:
from typing import Protocol
import unittest
from unittest.mock import Mock

class LoggerLike(Protocol):
    def log(self, message: str) -> None:
        ...

class Calculator:
    def __init__(self, logger: LoggerLike):
        self.logger = logger

    def add(self, a, b):
        result = a + b
        self.logger.log(f"Added {a} to {b} got {result}")
        return result

class TestBetterCalculator(unittest.TestCase):
    def test_addition(self):
        logger = Mock()
        calculator = Calculator(logger)

        result = calculator.add(1, 2)

        self.assertEqual(result, 3)
        calculator.logger.log.assert_called_once_with("Added 1 to 2 got 3")

unittest.main(defaultTest="TestBetterCalculator", argv=['-v'], exit=False)

Pokud `Mock` objekt není zavolán, unittest samozřejmě selže:

In [ ]:
from unittest.mock import Mock

mock = Mock(return_value=4)
result = mock(1, 2, 3)

try:
    mock.assert_called_with(1, 2)
except AssertionError as e:
    print(e)

##### Návratové hodnoty

- Mock objektům lze podstrčit i návratové hodnoty
- praktické např. v případech, kdy testovaná funkce závisí na výstupu z jiných objektů, které ovšem můžou být pomalé, nákladné na chod, nebo jejich chod může být závislý na vnějších faktorech (např. síťová komunikace)

In [ ]:
from unittest.mock import Mock

mock = Mock()
mock.return_value = 5

mock()

Případně pomocí `side_effect`, na který lze navázat existující funkci nebo např. výjimku.

In [ ]:
from unittest.mock import Mock

def some_side_effect():
    print("this is a side effect")

mock = Mock()
mock.side_effect = some_side_effect

mock()

In [ ]:
from unittest.mock import Mock

mock = Mock()
mock.side_effect = ZeroDivisionError("a specific Exception as a side effect")

try:
    mock()
except ZeroDivisionError as e:
    print(e)

#### Patching

Balík `unittest` nabízí několik možností jak dynamicky nahradit existující objekt (patching). Zkusme nahradit funkci `math.sqrt` funkcí `fake_sqrt`, která bude vracet dvojnásobek čísla místo skutečné odmocniny.

In [24]:
def fake_sqrt(x):
    return 2 * x

In [ ]:
import math
from unittest.mock import patch

@patch("math.sqrt")
def f(x, mock):
    mock.side_effect = fake_sqrt
    print(math.sqrt(x))

f(100)
print(math.sqrt(100))

In [25]:
import math
from unittest.mock import patch

with patch("math.sqrt") as mock:
    mock.side_effect = fake_sqrt
    print(math.sqrt(100))

print(math.sqrt(100))


200
10.0


In [ ]:
import math
from unittest.mock import patch

with patch("math.sqrt", side_effect=fake_sqrt):
    print(math.sqrt(100))

print(math.sqrt(100))

**Poznámka:** Balík `datetime` má svá specifika, která můžou komplikovat mocking. Přikládám odkaz na článek, který vysvětluje, co s tím: https://hakibenita.com/python-dependency-injection

## Test-driven development (TDD)

Základní cyklus:

1. napiš test
2. spusť test a nech ho selhat
3. napiš minimum kódu, aby prošel
4. refactor
5. opakuj

- TDD není o "testech po dopsání".
- TDD používá test jako nástroj návrhu.

### První test

In [26]:
import unittest

def normalize_name(name):
    raise NotImplementedError

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

E
ERROR: test_strips_whitespace (__main__.TestNormalizeName)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/730187432.py", line 8, in test_strips_whitespace
    self.assertEqual(normalize_name("  Alice  "), "Alice")
  File "/tmp/ipykernel_906449/730187432.py", line 4, in normalize_name
    raise NotImplementedError
NotImplementedError

----------------------------------------------------------------------
Ran 1 test in 0.003s

FAILED (errors=1)


In [27]:
# minimální implementace
def normalize_name(name):
    return name.strip()

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.003s

OK


### Druhý test

In [28]:
def normalize_name(name):
    return name.strip()

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob  "), "Alice Bob")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

F.
FAIL: test_collapses_internal_spaces (__main__.TestNormalizeName)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/1218134351.py", line 9, in test_collapses_internal_spaces
    self.assertEqual(normalize_name("Alice   Bob  "), "Alice Bob")
AssertionError: 'Alice   Bob' != 'Alice Bob'
- Alice   Bob
?       --
+ Alice Bob


----------------------------------------------------------------------
Ran 2 tests in 0.011s

FAILED (failures=1)


In [29]:
def normalize_name(name):
    return " ".join(name.strip().split())

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob  "), "Alice Bob")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


### Třetí test

In [ ]:
def normalize_name(name):
    return " ".join(name.strip().split())

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob"), "Alice Bob")

    def test_empty_name_raises(self):
        with self.assertRaises(ValueError):
            normalize_name("   ")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

In [ ]:
def normalize_name(name):
    normalized = " ".join(name.strip().split())
    if not normalized:
        raise ValueError("empty name")
    return normalized

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob"), "Alice Bob")

    def test_empty_name_raises(self):
        with self.assertRaises(ValueError):
            normalize_name("   ")

    def test_title_case(self):
        self.assertEqual(normalize_name("aLiCe boB"), "Alice Bob")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

In [ ]:
def normalize_name(name):
    normalized = " ".join(name.strip().split())
    if not normalized:
        raise ValueError("empty name")
    return normalized.title()

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob"), "Alice Bob")

    def test_empty_name_raises(self):
        with self.assertRaises(ValueError):
            normalize_name("   ")

    def test_title_case(self):
        self.assertEqual(normalize_name("aLiCe boB"), "Alice Bob")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

### Čtvrtý test

In [30]:
def normalize_name(name):
    normalized = " ".join(name.strip().split())
    if not normalized:
        raise ValueError("empty name")
    return normalized

class TestNormalizeName(unittest.TestCase):
    def test_strips_whitespace(self):
        self.assertEqual(normalize_name("  Alice  "), "Alice")

    def test_collapses_internal_spaces(self):
        self.assertEqual(normalize_name("Alice   Bob"), "Alice Bob")

    def test_empty_name_raises(self):
        with self.assertRaises(ValueError):
            normalize_name("   ")

    def test_title_case(self):
        self.assertEqual(normalize_name("aLiCe boB"), "Alice Bob")

unittest.main(defaultTest="TestNormalizeName", argv=[''], exit=False)

...F
FAIL: test_title_case (__main__.TestNormalizeName)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipykernel_906449/3198304888.py", line 19, in test_title_case
    self.assertEqual(normalize_name("aLiCe boB"), "Alice Bob")
AssertionError: 'aLiCe boB' != 'Alice Bob'
- aLiCe boB
+ Alice Bob


----------------------------------------------------------------------
Ran 4 tests in 0.021s

FAILED (failures=1)


## Příklady

### Pure function

In [31]:
def final_price():
    base = float(input("Base price: "))
    tax = 0.21
    print(base * (1 + tax))

final_price()

Base price:  25.5


30.855


Problémy:

- I/O (vstup od uživatele) namíchaný s logikou
- žádná návratová hodnota
- nelze otestovat bez patche funkce `input`
- side effects (print)

In [33]:
def final_price(base, tax=0.21):
    return base * (1 + tax)

In [34]:
import unittest

class TestFinalPrice(unittest.TestCase):
    def test_basic(self):
        self.assertAlmostEqual(final_price(100), 121.0)

    def test_custom_tax(self):
        self.assertAlmostEqual(final_price(100, 0.1), 110.0)

unittest.main(defaultTest="TestFinalPrice", argv=[''], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.007s

OK


### Dependency injection

In [40]:
class UserService:
    def get_upper_name(self, user_id):
        db = DatabaseClient()
        user = db.load_user(user_id)
        return user["name"].upper()

UserService().get_upper_name()

TypeError: UserService.get_upper_name() missing 1 required positional argument: 'user_id'

- třída si vytváří závislost interně
- při testování musíme patchovat

- lepší je, když závislost předáme zvenku (dependency injection)

In [35]:
class UserService:
    def __init__(self, database):
        self.database = database

    def get_upper_name(self, user_id):
        user = self.database.load_user(user_id)
        return user["name"].upper()

In [36]:
from unittest.mock import Mock
import unittest

class TestUserService(unittest.TestCase):
    def test_get_upper_name(self):
        db = Mock()
        db.load_user.return_value = {"name": "alice"}

        service = UserService(db)

        self.assertEqual(service.get_upper_name(1), "ALICE")
        db.load_user.assert_called_once_with(1)

unittest.main(defaultTest="TestUserService", argv=[''], exit=False)

.
----------------------------------------------------------------------
Ran 1 test in 0.004s

OK
